# 05 — Estabilidad de adquisición PCM

Preparación local para la campaña física de fase 5: **100 capturas
seriales**, cada una de 48 000 tramas PCM de cuatro canales. La campaña
comprueba la integridad de cada bloque, conserva las muestras y registra
duraciones de llamada y pausas entre llamadas. Se detiene en el primer
fallo y no reintenta.

Al activar `RUN_CAMPAIGN`, este notebook verifica el par BIT/HWH aprobado
y programa explícitamente el candidato. La configuración predeterminada
no toca la placa. Los tiempos del host no son marcas temporales acústicas;
las pausas entre llamadas no equivalen a muestras perdidas dentro de un
bloque válido y no demuestran adquisición continua. No se evalúan aquí
identidad física de micrófonos ni geometría.


In [ ]:
from pathlib import Path
import hashlib

from pynq_acoustic_camera import __version__, AcousticArray
from pynq_acoustic_camera.stability import run_pcm_stability_campaign

RUN_CAMPAIGN = False
BIT_FILE = Path("/home/xilinx/acoustic_camera_phase4_7/candidate/pdm_pcm_capture.bit")
EXPECTED_BIT_SHA256 = "5b7b14e34d9372302132746d9ad24475c4df9a1c66f06236ace752b439e91cf2"
EXPECTED_HWH_SHA256 = "fc757a78ae638b88eea46fa9a75a0fa67ecdaa09c2599aec19732102bdc9824f"
EXPECTED_BUILD_ID = "B691CC9C9F6249AD938D2063E4767C54"
EVIDENCE_ROOT = Path("/home/xilinx/acoustic_camera_phase5/evidence")

# Registrar condiciones de la sesión; no describir posiciones ni geometría.
OPERATOR_CONTEXT = {
    "acoustic_condition": "ambient",
    "board_power_source": None,
    "host_connection": None,
    "room_notes": None,
}


## Ejecución de la campaña

La celda siguiente queda inactiva mientras `RUN_CAMPAIGN` sea `False`.
Al habilitarla, primero compara ambos archivos con los hashes aprobados,
luego crea y descarga el overlay, verifica identidad y procedencia y
finalmente ejecuta la campaña. Cada bloque válido queda como `.npy` y
JSON. Si falla una captura o una validación, se conserva el diagnóstico,
se detiene la secuencia y se intenta apagar el reloj.


In [ ]:
if RUN_CAMPAIGN:
    if __version__ != "0.1.0rc2":
        raise RuntimeError("Instala la distribución 0.1.0rc2 antes de la campaña")
    hwh_file = BIT_FILE.with_suffix(".hwh")
    for path, expected in ((BIT_FILE, EXPECTED_BIT_SHA256),
                           (hwh_file, EXPECTED_HWH_SHA256)):
        actual = hashlib.sha256(path.read_bytes()).hexdigest()
        if actual != expected:
            raise ValueError("Hash de candidato incorrecto: " + str(path))

    from pynq import Overlay
    overlay = Overlay(str(BIT_FILE), download=True)
    array = AcousticArray.from_overlay(overlay, EXPECTED_BUILD_ID, provenance={
        "bit_sha256": EXPECTED_BIT_SHA256,
        "hwh_sha256": EXPECTED_HWH_SHA256,
        "bit_file": str(BIT_FILE),
        "hwh_file": str(hwh_file),
    })
    report = run_pcm_stability_campaign(
        array, EVIDENCE_ROOT, operator_context=OPERATOR_CONTEXT)
    print(report["status"], report["campaign_report_file"])
else:
    print("Campaña desactivada; no se programa ni accede a la placa.")
